In [1]:
# Install af3io from pip
!apt-get install --yes --quiet zstd
#!pip install --quiet af3io
!pip install --quiet git+https://github.com/jurgjn/af3io.git
!af3io --version

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (734 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/28

In [2]:
# Download one 5k pool from the MGen study
!curl -s https://zenodo.org/records/16920556/files/pools_5k.tar?download=1 \
| tar -xvf - --occurrence pools_5k_0040f80.zip

pools_5k_0040f80.zip


In [3]:
# Unpack & generate gzip/zstd baselines
!unzip pools_5k_0040f80.zip pools_5k_0040f80/pools_5k_0040f80_confidences.json
!gzip --keep pools_5k_0040f80/pools_5k_0040f80_confidences.json
!zstd --keep pools_5k_0040f80/pools_5k_0040f80_confidences.json
!ls -lhS pools_5k_0040f80/

Archive:  pools_5k_0040f80.zip
  inflating: pools_5k_0040f80/pools_5k_0040f80_confidences.json  
pools_5k_0040f80/pools_5k_0040f80_confidences.json :  4.45%   (234068369 => 10407605 bytes, pools_5k_0040f80/pools_5k_0040f80_confidences.json.zst) 
total 243M
-rw-r--r-- 1 root root 224M Jul 24  2025 pools_5k_0040f80_confidences.json
-rw-r--r-- 1 root root  10M Jul 24  2025 pools_5k_0040f80_confidences.json.zst
-rw-r--r-- 1 root root 9.3M Jul 24  2025 pools_5k_0040f80_confidences.json.gz


In [4]:
# Summary - how symmetric are contact_probs/pae?
!af3io confidences-show pools_5k_0040f80/pools_5k_0040f80_confidences.json

{
  "atom_chain_ids": ['B']*1961+['C']*3009+['D']*1258+['E']*5823+['F']*1438+['G']*484+['H']*909+['I']*2587+['J']*1012+['K']*950+['L']*1584+['M']*1338+['N']*1776+['O']*5025+['P']*4488+['Q']*882+['R']*4466+['S']*2272,
  "atom_plddts": <shape: (41262,) / min: 22.32 / max: 93.77 / nunique: 5,532>,
  "contact_probs": <shape: (5103, 5103) / min: 0.0 / max: 1.0 / nunique: 101 / 26,040,609 of 26,040,609 symmetric (100.0%)>,
  "pae": <shape: (5103, 5103) / min: 0.8 / max: 31.7 / nunique: 310 / 10,299,985 of 26,040,609 symmetric (39.6%)>,
  "token_chain_ids": ['A']*258+['B']*383+['C']*155+['D']*709+['E']*180+['F']*61+['G']*115+['H']*318+['I']*124+['J']*122+['K']*196+['L']*160+['M']*218+['N']*607+['O']*543+['P']*109+['Q']*561+['R']*284,
  "token_res_ids": chain(range(1,259),range(1,384),range(1,156),range(1,710),range(1,181),range(1,62),range(1,116),range(1,319),range(1,125),range(1,123),range(1,197),range(1,161),range(1,219),range(1,608),range(1,544),range(1,110),range(1,562),range(1,285))
}


In [5]:
# Compress confidences JSON into tweaked zarr
!af3io confidences-compress pools_5k_0040f80/pools_5k_0040f80_confidences.json

/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'zarr.json'
  return self._open_to_write(zinfo, force_zip64=force_zip64)


In [6]:
# Tweaked zarr is ~2x smaller than naive .gz/.zst
!ls -lhS pools_5k_0040f80/

total 247M
-rw-r--r-- 1 root root 224M Jul 24  2025 pools_5k_0040f80_confidences.json
-rw-r--r-- 1 root root  10M Jul 24  2025 pools_5k_0040f80_confidences.json.zst
-rw-r--r-- 1 root root 9.3M Jul 24  2025 pools_5k_0040f80_confidences.json.gz
-rw-r--r-- 1 root root 4.6M Feb 11 13:34 pools_5k_0040f80_confidences.zarr_compressed.zip


In [7]:
# Decompress tweaked zarr
!af3io confidences-decompress pools_5k_0040f80/pools_5k_0040f80_confidences.zarr_compressed.zip

Write: pools_5k_0040f80/pools_5k_0040f80_confidences.decompressed.json


In [8]:
# Test that decompression is byte-identical to original JSON
!md5sum pools_5k_0040f80/pools_5k_0040f80_confidences.json
!md5sum pools_5k_0040f80/pools_5k_0040f80_confidences.decompressed.json

6c0b6fe70145af41887a36f141863909  pools_5k_0040f80/pools_5k_0040f80_confidences.json
6c0b6fe70145af41887a36f141863909  pools_5k_0040f80/pools_5k_0040f80_confidences.decompressed.json


In [ ]:
# Clean up
!rm -rf pools_5k_0040f80